In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Hides annoying warning messages

# Load the dataset
# We use relative paths. Our notebook is in 'notebooks/', the data is in 'data/raw/'
# So we go up one folder level ('../') then into 'data/raw/'
file_path = '../data/raw/APL_Logistics.csv'

try:
    # We use 'latin-1' encoding because CSVs from legacy systems often have special characters
    df = pd.read_csv(file_path, encoding='latin-1')
    print("Dataset loaded successfully!")
    
    # Print the shape of the dataframe (Rows, Columns)
    print(f"The dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
    
except FileNotFoundError:
    print(f"Error: Could not find the file at {file_path}. Please check your folder structure.")

Dataset loaded successfully!
The dataset contains 180519 rows and 40 columns.


In [3]:
# Display the first 5 rows to see what the data looks like
display(df.head())

# Display a technical summary of the dataset
print("\n--- Dataset Info ---")
df.info()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortï¿½s,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortï¿½s,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class



--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Fname                 180519 non-null  str    
 12  Customer Id        

In [4]:
# Clean column names: lowercase, replace spaces with underscores, and remove parentheses
df.columns = (df.columns
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', ''))

# Let's check the new column names
print("Cleaned Column Names:")
print(list(df.columns))

Cleaned Column Names:
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_fname', 'customer_id', 'customer_lname', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_name', 'product_price', 'shipping_mode']


In [5]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")

# Check for missing values in each column
print("\n--- Missing Values per Column ---")
missing_values = df.isnull().sum()

# Only display columns that actually have missing values
missing_columns = missing_values[missing_values > 0]
if not missing_columns.empty:
    print(missing_columns)
else:
    print("No missing values found!")

Total duplicate rows: 0

--- Missing Values per Column ---
customer_lname      8
customer_zipcode    3
dtype: int64


In [6]:
# Create the Delivery Delay Gap feature
# Positive number = Delayed, Negative number = Early, 0 = On time
df['delay_gap'] = df['days_for_shipping_real'] - df['days_for_shipment_scheduled']

# Let's look at a quick summary of this new column
print("\n--- Delay Gap Summary Statistics ---")
print(df['delay_gap'].describe())


--- Delay Gap Summary Statistics ---
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: delay_gap, dtype: float64


In [7]:
# Drop the 11 rows that have missing values in specific columns
df.dropna(subset=['customer_lname', 'customer_zipcode'], inplace=True)

# Verify they are gone
print(f"Remaining rows after cleaning: {df.shape[0]}")

Remaining rows after cleaning: 180508


In [8]:
# Define a function to classify the delivery status based on the math
def classify_delivery(gap):
    if gap > 0:
        return 'Delayed'
    elif gap < 0:
        return 'Early'
    else:
        return 'On-time'

# Apply this function to create a new column
df['delivery_classification'] = df['delay_gap'].apply(classify_delivery)

# Check how many orders fall into each category
print("\n--- Delivery Classification Breakdown ---")
print(df['delivery_classification'].value_counts())


--- Delivery Classification Breakdown ---
delivery_classification
Delayed    103395
Early       43363
On-time     33750
Name: count, dtype: int64


In [9]:
# Save the cleaned dataframe to our processed folder
processed_path = '../data/processed/cleaned_apl_logistics.csv'

# index=False prevents pandas from writing row numbers as an extra column
df.to_csv(processed_path, index=False)

print(f"Cleaned dataset successfully saved to: {processed_path}")

Cleaned dataset successfully saved to: ../data/processed/cleaned_apl_logistics.csv
